# Estimating VMT by load zone region

This notebook processes HPMS 2023 data for each state in the WECC. The HPMS 2023 provides traffic data on all public roads in the US. We first calculate Heavy-Duty (HD) and Light-Duty (LD) VMT by road segment using direct AADT values or fallback estimates, then aggregate by load zone and state intersection.

Imports and configuration

In [3]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [4]:
# Define input and output paths
inputs_path = Path.cwd() / 'inputs'
hpms_dir = inputs_path / "HPMS_2023"

outputs_path = Path.cwd() / 'outputs'
outputs_path.mkdir(exist_ok=True)

# Define CRS
hpms_crs = "EPSG:4326"
projected_crs = "EPSG:5070"

# Read reference data for VMT per mi of road length by functional type and state
vmt_reference = pd.read_csv(inputs_path / "million_vmt_per_mi_road_length_by_functional_type.csv", index_col=0)

## 1: Calculate HD and Total VMT by road segment

Define a helper function for cleaning the data in a HPMS .gpkg dataset for one state. We standardize column names, remove duplicate road geometries, and remove roads without facility types (the latter represents a small proportion of all roads). 

In [10]:
def clean_columns(gdf):
    """
    Standardize column names and filter out invalid rows.
    """
    gdf.rename(
        columns={
            "Field6": "F_SYSTEM",  # Functional System (e.g., Interstate, Principal Arterial)
            "Field7": "FACILITY_TYPE",  # Facility Type (e.g., One-way, Two-way, Ramp)
            "Field14": "COUNTY_ID",
            "AADT_SINGLE_UNIT": "AADT_SINGL",
            "AADT_COMBINATION": "AADT_COMBI",
        },
        inplace=True,
    )

    # Clean up facility types: remove duplicate roads geometries (facility type 6) and those without facility types
    gdf["FACILITY_TYPE"] = (
        gdf["FACILITY_TYPE"].astype(str).str.strip().str.replace(".0", "", regex=False)
    )
    gdf = gdf[~gdf["FACILITY_TYPE"].isin(["6", "NULL"])].copy()

    # Drop rows with missing F_SYSTEM, and convert to str.
    gdf["F_SYSTEM"] = pd.to_numeric(gdf["F_SYSTEM"], errors="coerce")
    gdf.dropna(subset=["F_SYSTEM"], inplace=True)
    gdf["F_SYSTEM"] = gdf["F_SYSTEM"].astype(int).astype(str)

    # Assign unique segment IDs
    gdf["segment_id"] = range(len(gdf))
    return gdf


Define helper functions for computing the total VMT and heavy-duty VMT using the AADT and segment length for segments with defined AADT values. We use the total average annual daily traffic (AADT), average annual daily traffic for single-unit trucks (AADT_SINGL), and average annual daily traffic for combination-unit trucks (AADT_COMBI) for all roads in the WECC. 
- Total VMT for each road segment is calculated by: AADT * 365 * Segment Length (mi)
- Heavy-duty VMT for each road segment is calculated by: HD AADT * 365 * Segment Length (mi)
- HD AADT is calculated as the sum of AADT_COMBI and AADT_SINGL. 

In [11]:

def compute_total_vmt(gdf):
    """
    Compute Total VMT for segments with valid AADT.

    Returns:
    A copy of the input gdf, filtered features with valid AADT, with an added column for Total_VMT.
    A grouped gdf with total VMT and segment length by functional system
    """
    gdf = gdf[gdf["AADT"].astype(str) != "NULL"].copy()
    gdf[["AADT", "BeginPoint", "EndPoint"]] = gdf[
        ["AADT", "BeginPoint", "EndPoint"]
    ].astype(float)
    gdf["Total_VMT"] = gdf["AADT"] * gdf["segment_length"] * 365

    # Group-level totals by functional system
    grouped = gdf.groupby("F_SYSTEM").agg({"Total_VMT": "sum", "segment_length": "sum"})
    return gdf, grouped



def compute_hd_vmt(gdf):
    """
    Compute Heavy-Duty VMT for segments with valid truck counts.
    """
    gdf = gdf[
        (gdf["AADT_SINGL"].astype(str) != "NULL")
        & (gdf["AADT_COMBI"].astype(str) != "NULL")
    ].copy()
    gdf[["AADT_SINGL", "AADT_COMBI", "BeginPoint", "EndPoint"]] = gdf[
        ["AADT_SINGL", "AADT_COMBI", "BeginPoint", "EndPoint"]
    ].astype(float)
    gdf["AADTT"] = gdf["AADT_SINGL"] + gdf["AADT_COMBI"]
    gdf["HD_VMT"] = gdf["AADTT"] * gdf["segment_length"] * 365

    # Grouped for fallback estimation
    grouped = gdf.groupby("F_SYSTEM").agg({"HD_VMT": "sum", "segment_length": "sum"})
    return gdf, grouped


Since many road segments in the HPMS dataset are missing AADT values, we apply expansion factors to fill in missing VMT values for other road segments within the same functional system and state, using group averages for AADT, normalized by segment length, then multiplied by the length of each segment with missing AADT.

The expansion factor is defined as the ratio of the resulting road length with defined AADT values to the initial road length with the defined data. When the expansion factor > 10, we use data from the Highway Statistics 2023 for the VMT per mile for the corresponding state and functional type to fill in the missing VMT instead. This data is derived by dividing the VMT by functional class and state in the [Table VM-2](https://www.fhwa.dot.gov/policyinformation/statistics/2023/vm2.cfm) by the road system mileage by the corresponding functional class and state in [Table HM-20](https://www.fhwa.dot.gov/policyinformation/statistics/2023/hm20.cfm).

In [13]:
def fill_missing_vmt(gdf, grouped_totals, grouped_truck, state):
    for f in sorted(gdf["F_SYSTEM"].unique()):
        mask_tot = (gdf["F_SYSTEM"] == f) & gdf["Total_VMT"].isna()
        mask_hd = (gdf["F_SYSTEM"] == f) & gdf["HD_VMT"].isna()
        total_len = gdf.loc[gdf["F_SYSTEM"] == f, "segment_length"].sum()

        # Total VMT Fallback
        use_fallback = False
        try:
            known_len = grouped_totals.loc[f, "segment_length"]
            if known_len > 0:
                expansion_factor = total_len / known_len
                if expansion_factor > 10:
                    use_fallback = True
            else:
                use_fallback = True
        except:
            use_fallback = True

        if use_fallback:
            fallback_vmt = (
                vmt_reference.loc[state, f"FS_{int(f)}"] * 1e6
            )  # fallback from CSV
        else:
            fallback_vmt = (
                grouped_totals.loc[f, "Total_VMT"]
                / grouped_totals.loc[f, "segment_length"]
            )

        gdf.loc[mask_tot, "Total_VMT"] = (
            gdf.loc[mask_tot, "segment_length"] * fallback_vmt
        )

        # Heavy-Duty VMT Fallback 
        if f in ["6", "7"]:
            gdf.loc[mask_hd, "HD_VMT"] = 0
        else:
            try:
                fallback_hd = (
                    grouped_truck.loc[f, "HD_VMT"]
                    / grouped_truck.loc[f, "segment_length"]
                )
                gdf.loc[mask_hd, "HD_VMT"] = (
                    gdf.loc[mask_hd, "segment_length"] * fallback_hd
                )
            except:
                pass  # Leave HD_VMT as NaN if no data
    return gdf

Define a function that computes segment-level VMT, using the above helpers and looping through the HPMS 2023 dataset for each state.

In [14]:
def process_hpms_data():
    """
    Loop through each state's HPMS data and compute segment-level VMT,
    applying fallbacks when needed.
    """
    for file in hpms_dir.glob("*.gpkg"):
        print(f"\nProcessing {file.name}...")
        state = file.stem

        # Load raw data
        hpms = gpd.read_file(file)
        hpms = hpms.to_crs(projected_crs)

        # Standardize column names and remove facility types that we don't want
        hpms = clean_columns(hpms)

        # Compute length of each road segment (in miles, from HPMS)
        hpms["segment_length"] = hpms["EndPoint"] - hpms["BeginPoint"]

        # Compute Total and HD VMTs for segments with available data
        hpms_totals, grouped_totals = compute_total_vmt(hpms)
        hpms_truck, grouped_truck = compute_hd_vmt(hpms)

        # Join back segment-level VMT estimates
        hpms = hpms.merge(
            hpms_totals[["segment_id", "Total_VMT"]], on="segment_id", how="left"
        )
        hpms = hpms.merge(
            hpms_truck[["segment_id", "HD_VMT"]], on="segment_id", how="left"
        )

        # Fill missing VMT using grouped averages or reference estimates
        hpms = fill_missing_vmt(hpms, grouped_totals, grouped_truck, state)

        # Save processed file
        output_path = outputs_path / "HPMS_processed" 
        output_path.mkdir(exist_ok=True)
        
        hpms.to_file(output_path / f"{state}_vmt.gpkg", driver="GPKG", index=False)
        print(f"Saved: {output_path}")

Run the following cell to get cleaned data on HD_VMT and Total_VMT for all public road segments in the WECC.

In [20]:
process_hpms_data()


Processing SD.gpkg...


/opt/anaconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured MultiLineString' is converted to 'MultiLineString'
  return ogr_read(


Saved: /Users/nicholaskong/Desktop/REAM_lab/h2_demand_model/pre-processing/onroad_transport/outputs/HPMS_processed

Processing NE.gpkg...


/opt/anaconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured MultiLineString' is converted to 'MultiLineString'
  return ogr_read(


Saved: /Users/nicholaskong/Desktop/REAM_lab/h2_demand_model/pre-processing/onroad_transport/outputs/HPMS_processed

Processing CA.gpkg...


...

Saved: /Users/nicholaskong/Desktop/REAM_lab/h2_demand_model/pre-processing/onroad_transport/outputs/HPMS_processed

Processing AZ.gpkg...


/opt/anaconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured MultiLineString' is converted to 'MultiLineString'
  return ogr_read(


Saved: /Users/nicholaskong/Desktop/REAM_lab/h2_demand_model/pre-processing/onroad_transport/outputs/HPMS_processed

Processing CO.gpkg...


/opt/anaconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured MultiLineString' is converted to 'MultiLineString'
  return ogr_read(


Saved: /Users/nicholaskong/Desktop/REAM_lab/h2_demand_model/pre-processing/onroad_transport/outputs/HPMS_processed


## 2: Loop through all the state VMT files, correcting faulty VMT data and calculating LD and HD VMT by load zone within each state.  

LD VMT is then calculated by subtracting HD VMT from Total VMT. Segments with a negative LD VMT are corrected using the average % of total VMT that is light-duty within the same functional system and state. The LD VMT is set to the segment’s total VMT, multiplied by the aforementioned percentage. The HD VMT is set to the total VMT - LD VMT.

The HPMS 2023 dataset was then intersected with our custom load zones file to break roads segments down into segments that each fall within a load zone. VMT values for split segments are split proportional to segment length. LD and HD VMT values are then aggregated by load zone.

In [8]:
# Define new paths
cleaned_hpms_dir = outputs_path / "HPMS_processed" 
load_zone_path = inputs_path / "load_zones/load_zones.shp"

final_outputs_dir = outputs_path / "VMT_by_load_zone_within_state"
final_outputs_dir.mkdir(exist_ok=True)

In [9]:
# Load and prepare load zone geometries
load_zones = gpd.read_file(load_zone_path)
load_zones = load_zones.to_crs(projected_crs)
load_zones = load_zones[["LOAD_AREA", "geometry"]]

In [10]:
# Initialize lists for intermediate storage
summary_by_load_zone = []  # VMT summary by load zone
summary_by_state = []  # Summary of state totals and functional class breakdown

# Loop through all state-level HPMS files
for file in cleaned_hpms_dir.glob("*.gpkg"):
    print(f"\n📄 Processing {file.name}...")

    # Load and reproject HPMS data
    gdf = gpd.read_file(file)
    gdf = gdf.to_crs(projected_crs)

    # Compute light-duty VMT (LD_VMT)
    gdf["LD_VMT"] = gdf["Total_VMT"] - gdf["HD_VMT"]

    # Identify and count rows with bad (negative) LD_VMT
    neg_mask = gdf["LD_VMT"] < 0
    num_invalid = neg_mask.sum()
    print(f"Negative LD_VMT values: {num_invalid}")

    # Estimate fallback LD% share by F_SYSTEM using valid rows only
    valid = gdf[~neg_mask].copy()
    valid["LD_percent"] = valid["LD_VMT"] / valid["Total_VMT"]
    ld_share_by_fsystem = valid.groupby("F_SYSTEM")["LD_percent"].mean()

    # Fix rows with negative LD_VMT using fallback LD% values
    for f_system in gdf.loc[neg_mask, "F_SYSTEM"].unique():
        fs_mask = (gdf["F_SYSTEM"] == f_system) & neg_mask
        if f_system in ld_share_by_fsystem:
            ld_percent = ld_share_by_fsystem[f_system]
            gdf.loc[fs_mask, "LD_VMT"] = gdf.loc[fs_mask, "Total_VMT"] * ld_percent
            gdf.loc[fs_mask, "HD_VMT"] = gdf.loc[fs_mask, "Total_VMT"] * (
                1 - ld_percent
            )
        else:
            print(f"No fallback LD% for F_SYSTEM {f_system}. Skipping those rows.")

    # --- Create per-state summary row ---
    state_fips = int(gdf["StateID"].iloc[0])
    row = {
        "StateID": state_fips,
        "LD_VMT": gdf["LD_VMT"].sum(),
        "HD_VMT": gdf["HD_VMT"].sum(),
        "total_VMT": gdf["Total_VMT"].sum(),
    }

    # Convert F_SYSTEM to int (after fixing bad rows) for grouping
    gdf["F_SYSTEM"] = pd.to_numeric(gdf["F_SYSTEM"], errors="coerce")
    gdf.dropna(subset=["F_SYSTEM"], inplace=True)
    gdf["F_SYSTEM"] = gdf["F_SYSTEM"].astype(int)

    # Total VMT and length by functional system (for summary row)
    grouped = gdf.groupby("F_SYSTEM").agg({"Total_VMT": "sum", "segment_length": "sum"})
    for fs, data in grouped.iterrows():
        row[f"FS_{fs}_VMT"] = data["Total_VMT"]
        row[f"FS_{fs}_Length"] = data["segment_length"]

    summary_by_state.append(row)

    # Intersect roads with load zones 
    intersected = gpd.overlay(gdf, load_zones, how="intersection")

    # Compute segment lengths for the clipped geometries
    intersected["geometry_length_m"] = intersected.geometry.length

    # Weight each segment's contribution based on length fraction
    intersected["length_weight"] = intersected[
        "geometry_length_m"
    ] / intersected.groupby("segment_id")["geometry_length_m"].transform("sum")

    # Apply weights to VMT columns
    intersected["Total_VMT_weighted"] = (
        intersected["Total_VMT"] * intersected["length_weight"]
    )
    intersected["HD_VMT_weighted"] = (
        intersected["HD_VMT"] * intersected["length_weight"]
    )


    # --- Aggregation 1: VMT by load zone ---
    vmt_summary = intersected.groupby("LOAD_AREA")[
        ["Total_VMT_weighted", "HD_VMT_weighted"]
    ].sum()
    vmt_summary.rename(
        columns={"Total_VMT_weighted": "Total_VMT", "HD_VMT_weighted": "HD_VMT"},
        inplace=True,
    )
    vmt_summary["LD_VMT"] = vmt_summary["Total_VMT"] - vmt_summary["HD_VMT"]
    vmt_summary.reset_index(inplace=True)

    # Save state-level summary
    state_id = int(gdf["StateID"].iloc[0])
    output_path = final_outputs_dir / f"{state_id}_{file.name[:2]}.csv"
    vmt_summary.to_csv(output_path, index=False)
    print(f"Saved load zone summary: {output_path.name}")
    summary_by_load_zone.append(vmt_summary)


# Other exports for data checking

# Combined VMT by load zone across all states
summary_all = pd.concat(summary_by_load_zone, ignore_index=True)
summary_all.to_csv(final_outputs_dir / "load_zone_total_vmt_summary.csv", index=False)

# Combined state-level summary
state_summary_df = pd.DataFrame(summary_by_state).fillna(0)
state_summary_df = state_summary_df.sort_values("StateID")
state_summary_df.to_csv(final_outputs_dir / "vmt_summary_by_state.csv", index=False)


📄 Processing SD_vmt.gpkg...
Negative LD_VMT values: 1154
Saved load zone summary: 46_SD.csv

📄 Processing WY_vmt.gpkg...
Negative LD_VMT values: 178
Saved load zone summary: 56_WY.csv

📄 Processing OR_vmt.gpkg...
Negative LD_VMT values: 1623
Saved load zone summary: 41_OR.csv

📄 Processing CA_vmt.gpkg...
Negative LD_VMT values: 52230
Saved load zone summary: 6_CA.csv

📄 Processing NM_vmt.gpkg...
Negative LD_VMT values: 1500
Saved load zone summary: 35_NM.csv

📄 Processing AZ_vmt.gpkg...
Negative LD_VMT values: 941
Saved load zone summary: 4_AZ.csv

📄 Processing WA_vmt.gpkg...
Negative LD_VMT values: 2276
Saved load zone summary: 53_WA.csv

📄 Processing ID_vmt.gpkg...
Negative LD_VMT values: 2
Saved load zone summary: 16_ID.csv

📄 Processing NV_vmt.gpkg...
Negative LD_VMT values: 1022
Saved load zone summary: 32_NV.csv

📄 Processing MT_vmt.gpkg...
Negative LD_VMT values: 1464
Saved load zone summary: 30_MT.csv

📄 Processing UT_vmt.gpkg...
Negative LD_VMT values: 5416
Saved load zone su